# CPC353 Assignment 1 — Corpus preprocessing
**English–Malay instant noodle reviews · Python + pandas + NLTK**

Run the cells in order. This notebook prepares reusable data for **concordance, n-grams and stop-word analysis**. It does not train a classifier or infer language labels.

Original review text and metadata are preserved. Collected (`real`) and synthetic (`ai`) reviews have separate outputs. “Real” means self-collected, not independently verified. The snapshot is provisional: all 65 current reviews concern Shin Ramyun, while the study scope is instant noodles across products. Add other products with accurate source and product metadata, then rerun both notebooks.

**Unit of analysis:** one complete review. Ordered sentence token lists are also saved for later n-grams. The input Google Sheet and reference Colab are never edited.

The layout follows the supplied [reference Colab](https://colab.research.google.com/drive/1ykOKZZuW_DmxB8iLKAn9OT2E812foMCg): inspect → lowercase → clean → tokenize → filter → save. Classification and train/test splitting from that example do not apply to our three methods.

## 1. Set up
Use Python 3.10+ with `pandas` and `nltk`. NLTK's rule-based `RegexpTokenizer` needs **no resource downloads**.

Locally: `python -m pip install -r requirements.txt`

In Colab, run `%pip install pandas nltk openpyxl` in a separate cell if needed. Upload this repository's `data/raw/real_reviews.csv` and `ai_reviews.csv` to `/content/data/raw/`, or set `PROJECT_DIR` below to your project folder in mounted Google Drive. Files under `/content` disappear when the runtime resets; download the ZIP at the end or use a Drive folder.

In [1]:
from pathlib import Path
from collections import Counter
import csv
import hashlib
import html
import json
import platform
import re
import shutil
import unicodedata
import pandas as pd
import nltk
from nltk.tokenize import RegexpTokenizer

# Run from the repository root, or replace this with your project folder.
PROJECT_DIR = Path.cwd()
RAW_DIR = PROJECT_DIR / "data" / "raw"
OUTPUT_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional: path to a newly exported workbook; None uses the supplied CSV snapshot.
WORKBOOK = None  # Example: PROJECT_DIR / "corpus.xlsx"
print("Input:", RAW_DIR)
print("Output:", OUTPUT_DIR)

Input: C:\Users\USER\Desktop\Y3S1 CQ\CPC353\Assignment 1\CPC353-Assignment-1\data\raw
Output: C:\Users\USER\Desktop\Y3S1 CQ\CPC353\Assignment 1\CPC353-Assignment-1\data\processed


## 2. Read the two datasets
CSV snapshots keep this notebook easy to run without spreadsheet libraries. To use a fresh workbook export, set `WORKBOOK` above and install `openpyxl`. The workbook's headings are on row 4.

When you add real reviews, replace the real CSV with a new export (or use the workbook) and rerun **all cells**. Preserve review IDs, URLs, language labels and original wording. Do not include prompts as reviews.

In [2]:
input_files = []
tables = {}
for origin, sheet, filename in [
    ("real", "Self-Collected Data", "real_reviews.csv"),
    ("ai", "AI-Generated Data", "ai_reviews.csv"),
]:
    if WORKBOOK is None:
        path = RAW_DIR / filename
        table = pd.read_csv(path, dtype=str, keep_default_na=False)
    else:
        path = Path(WORKBOOK)
        table = pd.read_excel(path, sheet_name=sheet, header=3,
                              dtype=str, keep_default_na=False)
        table.insert(0, "source_row", range(5, len(table) + 5))
    input_files.append(path)
    table.columns = table.columns.str.strip()
    text_column = "Review Text" if origin == "real" else "Generated Review Text"
    required = {"Review ID", text_column, "Language Classification"}
    if not required.issubset(table.columns):
        raise ValueError(f"{origin}: missing headings {required - set(table.columns)}")
    tables[origin] = table

print(pd.DataFrame([
    {"origin": origin, "rows_loaded": len(table)}
    for origin, table in tables.items()
]).to_string(index=False))

origin  rows_loaded
  real           12
    ai           53


## 3. Standardize column names and preserve metadata
Only the **review-text column** is processed. IDs, origins, URLs, product names, ratings, assigned dates, language labels, switching labels and generation notes remain metadata.

Dates are not analysed; assigned real-review dates are synthetic, not publication dates. Existing language/switching annotations are reused, not treated as automatic proof of a switch. Product names alone do not establish code-switching.

In [3]:
frames = []
ignored_rows = {}
for origin, table in tables.items():
    text_column = "Review Text" if origin == "real" else "Generated Review Text"
    frame = table.copy()
    frame = frame.rename(columns={
        "Review ID": "review_id", text_column: "original_text",
        "Language Classification": "language",
        "Code-Switching Type": "switching_type",
        "Product / Eatery Name": "product",  # older CSV snapshots
        "Product / Brand Name": "product",  # current source sheet
    })
    # Ignore blank template rows and AI prompt-only rows.
    candidate = (frame["review_id"].str.strip().ne("") |
                 frame["original_text"].str.strip().ne(""))
    ignored_rows[origin] = int((~candidate).sum())
    frame = frame.loc[candidate].copy()
    frame["origin"] = origin
    frames.append(frame)
reviews = pd.concat(frames, ignore_index=True).fillna("")
print(reviews[["origin", "review_id", "language", "original_text"]].head(10).to_string(index=False))
print("Ignored non-review rows:", ignored_rows)

origin review_id      language                                                                                                                                                                                                                                                                                                                                                                                                                                                       original_text
  real    REV-01       English                                                                                                                                                                                                                                                                                                                           The noodles are thinner than other instant noodles. It comes with a sachet of duck seasoning, oil and chilli flakes. Taste is acceptable.
  real    REV-02       English    

## 4. Check missing data, duplicate IDs and duplicate text

The next three cells show separate diagnostic tables with clear headers: missing IDs/text, repeated IDs within an origin, and repeated review text after normalization. Each cell prints an explicit "none found" message when its table is empty. Review these tables before continuing. Structural problems are recorded in the audit and excluded as described below.

Text duplicates use Unicode, case, apostrophe and whitespace normalization. Cleaning keeps the first eligible text, checking collected data before AI copies. Repeated IDs within an origin are all held for manual resolution. If one review has more than one missing field, each missing field appears as a separate row in the missing-data table.


In [4]:
def normalize_text(text):
    return unicodedata.normalize("NFKC", text).translate(
        str.maketrans({"’": "'", "‘": "'", "–": "-", "—": "-"})
    ).casefold()

def text_hash(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

# Add entries only after checking the review. Example key:
# ("real", "REV-01", text_hash("exact original review text")): "Verified reason"
MANUAL_EXCLUSIONS = {}

reviews["text_sha256"] = reviews["original_text"].map(text_hash)
reviews["duplicate_key"] = reviews["original_text"].map(
    lambda text: re.sub(r"\s+", " ", normalize_text(text)).strip()
)
reviews["exclusion_reason"] = ""
for index, row in reviews.iterrows():
    missing_fields = []
    if not row["review_id"].strip():
        missing_fields.append("Missing review ID")
    if not row["original_text"].strip():
        missing_fields.append("Missing/empty review text")
    reviews.at[index, "exclusion_reason"] = "; ".join(missing_fields)
reviews["duplicate_of"] = ""
valid_ids = reviews["review_id"].str.strip().ne("")
duplicate_id_rows = reviews.loc[
    valid_ids & reviews.duplicated(["origin", "review_id"], keep=False)
].copy()
for row_index in duplicate_id_rows.index:
    reason = "Repeated ID within origin: resolve manually"
    existing = reviews.at[row_index, "exclusion_reason"]
    reviews.at[row_index, "exclusion_reason"] = "; ".join(
        value for value in [existing, reason] if value)

seen_text = {}
# Source loading order is real then AI. Original row order is preserved.
for index, row in reviews.iterrows():
    reason = MANUAL_EXCLUSIONS.get(
        (row["origin"], row["review_id"], row["text_sha256"]), "")
    if reason:
        reviews.at[index, "exclusion_reason"] = reason
    if reviews.at[index, "exclusion_reason"]:
        continue
    key = row["duplicate_key"]
    if key in seen_text:
        reviews.at[index, "exclusion_reason"] = "Duplicate normalized review text"
        reviews.at[index, "duplicate_of"] = seen_text[key]
    else:
        seen_text[key] = row["origin"] + ":" + row["review_id"]



In [5]:
# Missing values are listed by field so one review can show multiple problems.
missing_rows = []
for _, row in reviews.iterrows():
    missing_fields = []
    if not row["review_id"].strip():
        missing_fields.append("Review ID")
    if not row["original_text"].strip():
        missing_fields.append("Review Text")
    for field in missing_fields:
        missing_rows.append({
            "Origin": row["origin"],
            "Source Row": row.get("source_row", ""),
            "Review ID": row["review_id"] or "(blank)",
            "Missing Field": field,
            "Review Text": row["original_text"],
        })
missing_table = pd.DataFrame(missing_rows, columns=[
    "Origin", "Source Row", "Review ID", "Missing Field", "Review Text"])
print("MISSING REVIEW DATA")
if missing_table.empty:
    print("No missing review IDs or empty review text found.")
else:
    print(missing_table.to_string(index=False))


MISSING REVIEW DATA
No missing review IDs or empty review text found.


In [6]:
# A repeated ID is checked within its own origin (real or AI).
duplicate_id_table = (duplicate_id_rows.groupby(["origin", "review_id"], sort=False)
    .agg(Occurrences=("review_id", "size"),
         Source_Rows=("source_row", lambda values: ", ".join(map(str, values))))
    .reset_index()
    .rename(columns={"origin": "Origin", "review_id": "Duplicate Review ID",
                     "Source_Rows": "Source Rows"}))
print("DUPLICATE REVIEW IDS")
if duplicate_id_table.empty:
    print("No duplicate review IDs found.")
else:
    print(duplicate_id_table.to_string(index=False))


DUPLICATE REVIEW IDS
No duplicate review IDs found.


In [7]:
# Show every row in a repeated-text group, including the first occurrence.
duplicate_text_rows = reviews.loc[
    reviews["duplicate_key"].ne("") &
    reviews.duplicated("duplicate_key", keep=False)
].copy()
if duplicate_text_rows.empty:
    duplicate_text_table = pd.DataFrame(columns=[
        "Origin", "Review ID", "Source Row", "Duplicate Count", "Review Text"])
else:
    duplicate_text_rows["Duplicate Count"] = duplicate_text_rows.groupby(
        "duplicate_key")["duplicate_key"].transform("size")
    duplicate_text_table = duplicate_text_rows[[
        "origin", "review_id", "source_row", "Duplicate Count", "original_text"]
    ].rename(columns={"origin": "Origin", "review_id": "Review ID",
                      "source_row": "Source Row", "original_text": "Review Text"})
print("DUPLICATE REVIEW TEXT")
if duplicate_text_table.empty:
    print("No duplicate review text found.")
else:
    print(duplicate_text_table.to_string(index=False))


DUPLICATE REVIEW TEXT
No duplicate review text found.


In [8]:
reviews["included"] = reviews["exclusion_reason"].eq("")
audit = reviews.drop(columns="duplicate_key").copy()
cleaned = reviews.loc[reviews["included"]].drop(columns="duplicate_key").copy()
print("CLEANING DECISIONS BY ORIGIN")
print(audit.groupby(["origin", "included"]).size().to_string())


CLEANING DECISIONS BY ORIGIN
origin  included
ai      True        53
real    True        12


## 5. Flag metadata and source checks
These checks **flag** rows for the group; they do not guess new language labels or silently change sources. No website verification is performed here.

In the current snapshot REV-01 mentions duck seasoning despite its recorded Shin Ramyun product metadata. It needs a source/content check before final findings. We keep it in the preprocessing output with a visible warning; an inclusion count is not a source-verification count. Review this queue before final analysis.

In [9]:
VALID_LANGUAGES = {"English", "Malay", "Code-switched", "Other/uncertain"}

def review_flags(row):
    flags = []
    if row["language"] not in VALID_LANGUAGES:
        flags.append("Missing/unrecognized language annotation")
    if not row.get("product", "").strip():
        flags.append("Missing product metadata")
    if row["origin"] == "real":
        if not re.match(r"^https?://[^/\s]+", row.get("Source URL", "").strip()):
            flags.append("Missing/invalid source URL")
        if "duck seasoning" in row["original_text"].casefold():
            flags.append("Check seasoning description against recorded product/source")
    notes = row.get("Notes / Annotator Comments", "")
    if re.search(r"manual review|verify source|replace or exclude|flag for", notes, re.I):
        flags.append("Unresolved source annotation: " + notes)
    return "; ".join(flags)

audit["review_flags"] = audit.apply(review_flags, axis=1)
cleaned["review_flags"] = cleaned.apply(review_flags, axis=1)
manual_queue = audit.loc[audit["review_flags"].ne("")].copy()
print("Rows needing manual checks:", len(manual_queue))
print(manual_queue[["origin", "review_id", "review_flags"]].to_string(index=False))

Rows needing manual checks: 1
origin review_id                                                review_flags
  real    REV-01 Check seasoning description against recorded product/source


## 6. Normalize a derived copy
Keep `original_text` intact. The derived text uses Unicode normalization, lowercase, straight apostrophes, HTML entity decoding, and removes URLs, email addresses and HTML tags. Whitespace is collapsed.

No translation, spelling correction, abbreviation expansion, stemming or lemmatization: these could erase informal English–Malay patterns. Repeated letters and forms such as `sgt`, `tak`, `lah` and `panas2` remain distinct.

Punctuation is retained in `normalized_text` for context and approximate sentence splitting. The word-token view excludes punctuation, emojis and standalone numbers. Inspect originals for price, emoji and punctuation meanings.

In [10]:
def clean_text(text):
    text = normalize_text(html.unescape(text))
    text = re.sub(r"https?://\S+|www\.\S+|\b[^\s@]+@[^\s@]+\b", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", text).strip()

cleaned["normalized_text"] = cleaned["original_text"].map(clean_text)
print(cleaned[["review_id", "original_text", "normalized_text"]].head(2).to_string(index=False))

review_id                                                                                                                                                                                                                          original_text                                                                                                                                                                                                                        normalized_text
   REV-01                                                                                              The noodles are thinner than other instant noodles. It comes with a sachet of duck seasoning, oil and chilli flakes. Taste is acceptable.                                                                                              the noodles are thinner than other instant noodles. it comes with a sachet of duck seasoning, oil and chilli flakes. taste is acceptable.
   REV-02 The flavour of this Nongshim Shin Ramyun is go

## 7. Tokenize with NLTK and preserve boundaries
`RegexpTokenizer` keeps Unicode letter-based words, internal apostrophes/hyphens, and trailing digits (`panas2`). Negated contractions such as `don't` and `can't` remain whole.

Sentence boundaries use a deliberately simple punctuation rule (`.`, `!`, `?`, `…`). It may oversplit abbreviations; check concordance against original text. Separate sentence arrays prevent later n-grams from crossing those boundaries. Reviews are never joined together.

In [11]:
TOKEN_PATTERN = r"[^\W\d_]+(?:['-][^\W\d_]+)*\d*"
tokenizer = RegexpTokenizer(TOKEN_PATTERN, flags=re.UNICODE)

def sentence_tokens(text):
    parts = re.split(r"[.!?…]+", text)
    return [tokens for part in parts if (tokens := tokenizer.tokenize(part))]

cleaned["sentence_tokens_before"] = cleaned["normalized_text"].map(sentence_tokens)
cleaned["tokens_before"] = cleaned["sentence_tokens_before"].map(
    lambda sentences: [token for sentence in sentences for token in sentence]
)
cleaned["word_text"] = cleaned["tokens_before"].map(" ".join)
print(cleaned[["review_id", "tokens_before"]].head(3).to_string(index=False))

review_id                                                                                                                                                                                                                                                                                                                                                      tokens_before
   REV-01                                                                                                                                                                                                      [the, noodles, are, thinner, than, other, instant, noodles, it, comes, with, a, sachet, of, duck, seasoning, oil, and, chilli, flakes, taste, is, acceptable]
   REV-02                                                                                      [the, flavour, of, this, nongshim, shin, ramyun, is, good, though, sometimes, the, seasoning, and, dry, condiment, packs, tend, to, be, a, little, stale, the, powder, seasonin

## 8. Define a small, inspectable bilingual stop-word list
This is a **project-authored conservative list**, not an official Malay list or NLTK's English list. NLTK provides the tokenizer. The list follows the earlier stop-word script's grammatical-word approach and adds documented Malay abbreviations.

Retain negation, contrast, intensifiers and discourse particles: e.g. `not`, `tak`, `bukan`, `but`, `tapi`, `very`, `sangat`, `lah`, `kan`. Keep brand and attribute words too. Use the same list for both origins. Review it manually before reporting; changing the list requires rerunning all outputs.

Reference for the filtering method: [NLTK Book, Chapter 2](https://www.nltk.org/book/ch02.html). Malay entries and protection choices are our project decisions, not a validated external lexicon.

In [12]:
ENGLISH = set("""
a an the i me my mine we us our ours you your yours he him his
she her hers it its they them their theirs this that these those am is are was
were be been being have has had do does did of to in on at by for from with
as and or if than then there here which who whom whose what when where how
""".split())
MALAY = set("""
saya aku kami kita anda awak kamu dia mereka ini itu yang dan
atau di ke dari daripada kepada pada dalam dengan untuk sebagai ialah adalah
""".split())
INFORMAL = {"yg": "yang", "dgn": "dengan", "dlm": "dalam",
            "ni": "ini", "tu": "itu"}
PROTECTED = set("""
not no nor never neither without cannot tak tidak bukan belum jangan tiada
tanpa kurang x xde takde takkan janganlah but tapi tetapi namun
lah la leh lor meh kan pun je jer sangat too very
""".split())
STOPWORDS = (ENGLISH | MALAY | set(INFORMAL)) - PROTECTED
stopword_rows = [
    {"word": word, "language": "English" if word in ENGLISH else "Malay",
     "reason": ("Informal variant of " + INFORMAL[word]) if word in INFORMAL
               else "Project-selected grammatical/function word"}
    for word in sorted(STOPWORDS)
]
stopword_table = pd.DataFrame(stopword_rows)
print("Stop words:", len(STOPWORDS))
print(stopword_table.to_string(index=False))

Stop words: 100
    word language                                     reason
       a  English Project-selected grammatical/function word
  adalah    Malay Project-selected grammatical/function word
     aku    Malay Project-selected grammatical/function word
      am  English Project-selected grammatical/function word
      an  English Project-selected grammatical/function word
     and  English Project-selected grammatical/function word
    anda    Malay Project-selected grammatical/function word
     are  English Project-selected grammatical/function word
      as  English Project-selected grammatical/function word
      at  English Project-selected grammatical/function word
    atau    Malay Project-selected grammatical/function word
    awak    Malay Project-selected grammatical/function word
      be  English Project-selected grammatical/function word
    been  English Project-selected grammatical/function word
   being  English Project-selected grammatical/function word
      by

## 9. Save a filtered view alongside the unfiltered view
Stop-word removal is a separate, reversible view. Removed tokens are saved too. Do not use `tokens_after` for ordinary n-grams: filtering can join words that were never adjacent. Use `sentence_tokens_before` for n-grams, and original/normalized text for concordance.

In [13]:
cleaned["tokens_after"] = cleaned["tokens_before"].map(
    lambda tokens: [token for token in tokens if token not in STOPWORDS])
cleaned["tokens_removed"] = cleaned["tokens_before"].map(
    lambda tokens: [token for token in tokens if token in STOPWORDS])
cleaned["filtered_text"] = cleaned["tokens_after"].map(" ".join)
cleaned["token_count_before"] = cleaned["tokens_before"].map(len)
cleaned["token_count_after"] = cleaned["tokens_after"].map(len)
cleaned["removed_count"] = cleaned["tokens_removed"].map(len)
print(cleaned[["review_id", "word_text", "filtered_text"]].head(3).to_string(index=False))

review_id                                                                                                                                                                                                                                                                                                       word_text                                                                                                                                                                                                                                           filtered_text
   REV-01                                                                                                                                                                           the noodles are thinner than other instant noodles it comes with a sachet of duck seasoning oil and chilli flakes taste is acceptable                                                                                                                             

## 10. Summarize cleaning and language coverage
Report counts separately for each origin, including exclusions and rows needing checks. The code-switched subset is the relevant starting point for the research question; all-language views are background.

The brief targets approximately 100–200 reviews. This is a preliminary snapshot, so the notebook warns when fewer than 100 reviews remain. Frequencies and attributes have not been interpreted at this stage.

In [14]:
summary_rows = []
for origin in ["real", "ai"]:
    before = audit.loc[audit["origin"].eq(origin)]
    retained = cleaned.loc[cleaned["origin"].eq(origin)]
    summary_rows.append({
        "origin": origin, "reviews_before": len(before),
        "reviews_retained": len(retained),
        "reviews_excluded": len(before) - len(retained),
        "needs_manual_check": int(retained["review_flags"].ne("").sum()),
        "tokens_before": int(retained["token_count_before"].sum()),
        "tokens_after": int(retained["token_count_after"].sum()),
    })
summary = pd.DataFrame(summary_rows)
language_counts = (cleaned.groupby(["origin", "language"]).size()
                  .reset_index(name="reviews"))
print(summary.to_string(index=False))
print(language_counts.to_string(index=False))
if len(cleaned) < 100:
    print(f"PROVISIONAL: {len(cleaned)} retained reviews; collect more toward the 100–200 target.")

origin  reviews_before  reviews_retained  reviews_excluded  needs_manual_check  tokens_before  tokens_after
  real              12                12                 0                   1            445           307
    ai              53                53                 0                   0            756           588
origin      language  reviews
    ai Code-switched       20
    ai       English       16
    ai         Malay       17
  real Code-switched        5
  real       English        7
PROVISIONAL: 65 retained reviews; collect more toward the 100–200 target.


## 11. Validate the result
These checks guard against lost original text, removed negations, origin mixing and changed token order. Small examples also exercise empty text, duplicates and tokenization edge cases.

In [15]:
assert audit["origin"].isin(["real", "ai"]).all()
assert not cleaned.duplicated(["origin", "review_id"]).any()
assert not cleaned["original_text"].str.strip().eq("").any()
assert cleaned["text_sha256"].equals(cleaned["original_text"].map(text_hash))
assert not (STOPWORDS & PROTECTED)
assert tokenizer.tokenize(clean_text("Not sedap, tak mahal! Bukan bad; don't remove panas2.")) == [
    "not", "sedap", "tak", "mahal", "bukan", "bad", "don't", "remove", "panas2"]
assert tokenizer.tokenize(clean_text("See https://example.com or a@b.com 😊")) == ["see", "or"]
assert sentence_tokens(clean_text("Sedap! Not cheap.")) == [["sedap"], ["not", "cheap"]]
assert clean_text("  TASTE  ") == clean_text("taste")
for row in cleaned.itertuples():
    assert row.tokens_before == [t for sent in row.sentence_tokens_before for t in sent]
    assert row.tokens_after == [t for t in row.tokens_before if t not in STOPWORDS]
    assert len(row.tokens_before) == len(row.tokens_after) + len(row.tokens_removed)
    assert all(t in row.tokens_after for t in row.tokens_before if t in PROTECTED)
print("Validation passed.")

Validation passed.


## 12. Export reusable files
JSON Lines (`.jsonl`) keeps actual token arrays. CSV stores arrays as JSON strings; restore them with `json.loads`, not `eval`.

Outputs: separate real/AI datasets and code-switched subsets; audit and manual-check queue; before/after and language counts; stop-word/protected lists; and a manifest with input hashes, versions and decisions. Rerunning replaces outputs in this folder. A ZIP is created for download or later reuse.

In [16]:
LIST_COLUMNS = ["tokens_before", "sentence_tokens_before", "tokens_after", "tokens_removed"]

def export_dataset(frame, name):
    frame.to_json(OUTPUT_DIR / f"{name}.jsonl", orient="records",
                  lines=True, force_ascii=False)
    csv_frame = frame.copy()
    for column in LIST_COLUMNS:
        csv_frame[column] = csv_frame[column].map(
            lambda value: json.dumps(value, ensure_ascii=False))
    csv_frame.to_csv(OUTPUT_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")

for origin in ["real", "ai"]:
    group = cleaned.loc[cleaned["origin"].eq(origin)].copy()
    export_dataset(group, f"{origin}_preprocessed")
    export_dataset(group.loc[group["language"].eq("Code-switched")],
                   f"{origin}_codeswitched")

audit.to_csv(OUTPUT_DIR / "review_audit.csv", index=False, encoding="utf-8-sig")
manual_queue.to_csv(OUTPUT_DIR / "manual_review_queue.csv", index=False, encoding="utf-8-sig")
summary.to_csv(OUTPUT_DIR / "cleaning_summary.csv", index=False)
language_counts.to_csv(OUTPUT_DIR / "language_counts.csv", index=False)
stopword_table.to_csv(OUTPUT_DIR / "stopwords_used.csv", index=False)
pd.DataFrame({"word": sorted(PROTECTED)}).to_csv(
    OUTPUT_DIR / "protected_words.csv", index=False)

manifest = {
    "preprocessing_version": "1.0",
    "python": platform.python_version(), "pandas": pd.__version__, "nltk": nltk.__version__,
    "input_sha256": {str(path.relative_to(PROJECT_DIR)) if path.is_relative_to(PROJECT_DIR)
                     else path.name: hashlib.sha256(path.read_bytes()).hexdigest()
                     for path in dict.fromkeys(input_files)},
    "unit": "complete review; sentence arrays retained separately",
    "token_pattern": TOKEN_PATTERN,
    "sentence_rule": "Split normalized text on runs of . ! ? or ellipsis; approximate",
    "deduplication": "NFKC, casefold, normalized apostrophes/dashes and whitespace; real first",
    "ignored_nonreview_rows": ignored_rows,
    "counts": summary.to_dict(orient="records"),
    "stopwords": sorted(STOPWORDS), "protected_words": sorted(PROTECTED),
    "status": "Provisional. Resolve source flags and annotation checks before final findings.",
    "dates": "Metadata only. Assigned real dates are synthetic, not publication dates.",
    "ai_provenance": "Theme-based synthetic examples; no direct real-review seed links claimed.",
    "limits": ["No automatic language detection", "No stemming or spelling correction",
               "No source-site verification", "No fuzzy duplicate removal",
               "Emoji and standalone numbers absent from lexical tokens; originals retained"],
}
(OUTPUT_DIR / "run_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

# Check saved arrays and text survive a round trip.
for origin in ["real", "ai"]:
    path = OUTPUT_DIR / f"{origin}_preprocessed.jsonl"
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]
    expected = cleaned.loc[cleaned["origin"].eq(origin)]
    assert [r["original_text"] for r in rows] == expected["original_text"].tolist()
    assert [r["tokens_before"] for r in rows] == expected["tokens_before"].tolist()
    assert all(r["origin"] == origin for r in rows)

archive = shutil.make_archive(str(PROJECT_DIR / "preprocessed_results"), "zip", OUTPUT_DIR)
print("Saved", len(list(OUTPUT_DIR.iterdir())), "output files.")
print("Downloadable archive:", archive)

Saved 15 output files.
Downloadable archive: C:\Users\USER\Desktop\Y3S1 CQ\CPC353\Assignment 1\CPC353-Assignment-1\preprocessed_results.zip


## 13. Load the result in a later analysis notebook
No need to repeat preprocessing in each analysis. Reload JSONL for list-valued tokens. Keep source labels and original texts in the saved files; combine included reviews for corpus-wide analysis. Use the code-switched subsets when examining switching context.

**Next:** inspect the manual-check queue, collect more verified reviews, review language labels and the stop-word list, then rerun. Final concordance, n-gram tables and stop-word comparisons belong in the later analysis notebooks.

In [17]:
real_ready = pd.read_json(OUTPUT_DIR / "real_preprocessed.jsonl", lines=True)
ai_ready = pd.read_json(OUTPUT_DIR / "ai_preprocessed.jsonl", lines=True)
print("Ready for later analysis:", len(real_ready), "collected and", len(ai_ready), "AI reviews")
print("Concordance: original_text or normalized_text")
print("N-grams: sentence_tokens_before (unfiltered; keep each review separate)")
print("Stop-word comparison: tokens_before, tokens_after, tokens_removed")

# In Colab only, uncomment to download before ending the session:
# from google.colab import files
# files.download(str(PROJECT_DIR / "preprocessed_results.zip"))

Ready for later analysis: 12 collected and 53 AI reviews
Concordance: original_text or normalized_text
N-grams: sentence_tokens_before (unfiltered; keep each review separate)
Stop-word comparison: tokens_before, tokens_after, tokens_removed
